<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Developing Applications with Apache Spark</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">3. DataFrame Relational Operations</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</p>

# 3.1 Lecture - DataFrame Relational Operations
### Duration - 27 Minutes

> *Combining distributed datasets with joins and set operations*

This lecture covers the relational side of the DataFrame API: join types and conditions, set operations for comparing datasets, and the performance practices that keep joins fast on distributed data.

## Learning Objectives

**By the end of this lecture, you will be able to:**

- Describe different types of DataFrame joins (inner, outer, left, right, cross) and their applications in combining datasets
- Use set operations like union, intersect, and subtract for combining or finding differences between DataFrames
- Apply best practices for optimizing relational operations, including choosing appropriate join strategies and handling data skew

## A. DataFrame Join Operations

Joins combine two DataFrames based on matching key column(s):

- Multiple join types supported: inner, outer, left, right, cross
- The join condition can use single or multiple columns
- Column naming conflicts are handled with DataFrame aliases or by renaming columns before the join

| Join Type (`how=`) | Returns |
|--------------------|---------|
| __`inner`__ (default) | Rows with matching keys in both DataFrames |
| __`left`__ | All left rows, with nulls where the right side has no match |
| __`right`__ | All right rows, with nulls where the left side has no match |
| __`full_outer`__ | All rows from both sides, with nulls for missing matches |
| __`left_semi`__ | Left rows that have a match (no right columns) |
| __`left_anti`__ | Left rows that have no match |
| __`cross`__ | Every combination of rows (use with care) |

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Anatomy of a distributed join</strong>
      </div>
    </div>
  </summary>
  <div style="width:100%;max-width:820px;margin:14px auto"><svg viewBox="0 0 1120 360" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Anatomy of a join" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block"><title>Anatomy of a join</title><defs><marker id="ah_asp2_join" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,1 L10,5 L0,9 z" fill="#3C4B52"/></marker></defs><rect x="40" y="40" width="300" height="110" rx="10" fill="#DCE0E2" stroke="#618794" stroke-width="2"/><text x="190.0" y="100.9" text-anchor="middle" font-size="21" fill="#0B2026" font-weight="700">users DataFrame</text><rect x="40" y="210" width="300" height="110" rx="10" fill="#DCE0E2" stroke="#618794" stroke-width="2"/><text x="190.0" y="270.9" text-anchor="middle" font-size="21" fill="#0B2026" font-weight="700">orders DataFrame</text><polygon points="512,120 648,120 690,180.0 648,240 512,240 470,180.0" fill="#FFF3E0" stroke="#FF9800" stroke-width="2"/><text x="580.0" y="187.0" text-anchor="middle" font-size="20" font-weight="700" fill="#0B2026">join on user_id</text><rect x="800" y="120" width="290" height="120" rx="10" fill="#E8F5E9" stroke="#4CAF50" stroke-width="2"/><text x="945.0" y="175.7" text-anchor="middle" font-size="21" fill="#0B2026" font-weight="700">Combined DataFrame</text><text x="945.0" y="199.4" text-anchor="middle" font-size="16" fill="#4A5A62" font-style="italic">user and order columns</text><path d="M340,95 L476,154.0" fill="none" stroke="#3C4B52" stroke-width="2.2" marker-end="url(#ah_asp2_join)"/><path d="M340,265 L476,206.0" fill="none" stroke="#3C4B52" stroke-width="2.2" marker-end="url(#ah_asp2_join)"/><path d="M690,180.0 L800,180.0" fill="none" stroke="#3C4B52" stroke-width="2.2" marker-end="url(#ah_asp2_join)"/></svg></div>
</details>

### join Examples

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># Basic join example</span>
<span style="color: #CCC">df1</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(df2,</span> <span style="color: #7EC699">"user_id"</span><span style="color: #CCC">)</span>  <span style="color: #999"># Inner join by default</span>
<span></span>
<span style="color: #999"># Specifying join type</span>
<span style="color: #CCC">df1</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(df2,</span>
    <span style="color: #CCC">on</span><span style="color: #67CDCC">=</span><span style="color: #CCC">df1</span><span style="color: #67CDCC">.</span><span style="color: #CCC">user_id</span> <span style="color: #67CDCC">==</span> <span style="color: #CCC">df2</span><span style="color: #67CDCC">.</span><span style="color: #CCC">user_id,</span>
    <span style="color: #CCC">how</span><span style="color: #67CDCC">=</span><span style="color: #7EC699">"left"</span><span style="color: #CCC">)</span>
<span></span>
<span style="color: #999"># Multiple conditions with DataFrame aliases</span>
<span style="color: #CC99CD">from</span><span style="color: #CCC"> pyspark.sql.functions </span><span style="color: #CC99CD">import</span> <span style="color: #CCC">col</span>
<span style="color: #CCC">users</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"u"</span><span style="color: #CCC">)</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(</span>
    <span style="color: #CCC">orders</span><span style="color: #67CDCC">.</span><span style="color: #CCC">alias(</span><span style="color: #7EC699">"o"</span><span style="color: #CCC">),</span>
    <span style="color: #CCC">[col(</span><span style="color: #7EC699">"u.id"</span><span style="color: #CCC">)</span> <span style="color: #67CDCC">==</span> <span style="color: #CCC">col(</span><span style="color: #7EC699">"o.user_id"</span><span style="color: #CCC">),</span>
     <span style="color: #CCC">col(</span><span style="color: #7EC699">"u.region"</span><span style="color: #CCC">)</span> <span style="color: #67CDCC">==</span> <span style="color: #CCC">col(</span><span style="color: #7EC699">"o.region"</span><span style="color: #CCC">)],</span>
    <span style="color: #7EC699">"inner"</span><span style="color: #CCC">)</span></code></pre>



## B. Set Operations

Set operations treat DataFrames as sets of rows, helping you understand relationships within datasets. Given `A = [1, 2, 3]` and `B = [2, 3, 4]`:

- **UNION** - includes all unique elements from both sets, no duplicates: `A UNION B -> [1, 2, 3, 4]`
- **INTERSECTION** - includes elements in both sets: `A INTERSECT B -> [2, 3]`
- **SUBTRACTION** - includes elements in A only: `A MINUS B -> [1]`

All DataFrame set operations require matching schemas. The methods:

| Operation | Description | Example |
|-----------|-------------|---------|
| __`union()`__ | Combines all rows from both DataFrames by position | `df1.union(df2)` |
| __`unionByName()`__ | Combines DataFrames matching by column names | `df1.unionByName(df2)` |
| __`intersect()`__ | Returns only rows present in both DataFrames | `df1.intersect(df2)` |
| __`subtract()`__ | Returns rows in the first DataFrame but not in the second | `df1.subtract(df2)` |

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#8505;&#65039;</span>
<div>
<strong style="color: #0d47a1; font-size: 1.1em">Duplicates</strong>
<p style="margin: 8px 0 0 0; color: #333"><code>union()</code> in Spark behaves like SQL <code>UNION ALL</code> - it keeps duplicates (deduplicate with <code>distinct()</code>). <code>intersect()</code> and <code>subtract()</code> remove duplicates; use the <code>intersectAll()</code> and <code>exceptAll()</code> variants to preserve them.</p>
</div>
</div>
</div>

## C. Join Performance Considerations

<p style="font-size: 1em; line-height: 1.6; color: #333">Optimizing relational operations in a distributed environment:</p>

- **Join strategy selection** - Spark automatically chooses the join strategy; reference the smaller DataFrame first where practical
- **Broadcast joins** - Spark can avoid shuffling the large side entirely by copying a small DataFrame to every Executor; use the `broadcast()` hint to request this explicitly

<pre style="background:#1e1e1e;color:#cccccc;border-radius:8px;padding:16px;margin:16px 0;overflow-x:auto;font-family:Consolas,Monaco,monospace;font-size:14px;line-height:1.5"><code><span style="color: #999"># Good - small_df is smaller</span>
<span style="color: #CCC">small_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(large_df,</span> <span style="color: #7EC699">"key"</span><span style="color: #CCC">)</span>
<span></span>
<span style="color: #999"># Less efficient - large_df is larger</span>
<span style="color: #CCC">large_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(small_df,</span> <span style="color: #7EC699">"key"</span><span style="color: #CCC">)</span>
<span></span>
<span style="color: #999"># Broadcasting a small DataFrame</span>
<span style="color: #CC99CD">from</span><span style="color: #CCC"> pyspark.sql.functions </span><span style="color: #CC99CD">import</span> <span style="color: #CCC">broadcast</span>
<span style="color: #CCC">large_df</span><span style="color: #67CDCC">.</span><span style="color: #CCC">join(broadcast(small_df),</span> <span style="color: #7EC699">"key"</span><span style="color: #CCC">)</span></code></pre>



### More Join Performance Considerations

- **Data skew handling**
  - Uneven distribution of join keys can impact performance
  - Consider repartitioning in some cases
- **Memory management**
  - Monitor shuffle spill metrics for joins
  - Consider caching frequently joined DataFrames
  - Use projection to select only needed columns before joining

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
<div style="display: flex; align-items: flex-start; gap: 12px">
<span style="font-size: 24px">&#128161;</span>
<div>
<strong style="color: #00695c; font-size: 1.1em">Use the Spark UI</strong>
<p style="margin: 8px 0 0 0; color: #333">Use the Spark UI to monitor join performance and identify bottlenecks - the SQL tab shows the chosen join strategy (e.g. <code>BroadcastHashJoin</code> vs <code>SortMergeJoin</code>) and per-stage shuffle metrics.</p>
</div>
</div>
</div>

## Summary

- Joins combine DataFrames on key columns; choose the join type (`inner`, `left`, `right`, `full_outer`, `semi`, `anti`, `cross`) based on which unmatched rows you need
- Set operations (`union()`, `unionByName()`, `intersect()`, `subtract()`) compare whole rows and require matching schemas
- Broadcast small DataFrames to avoid shuffling large ones; project columns before joining; watch for data skew
- The Spark UI shows which join strategy the optimizer picked and where the time went

## Knowledge Check

Answer the questions below to check your understanding of this lecture.

<div style="font-family:-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,sans-serif;width:100%;margin:10px 0;padding:24px;background:#f5f7fa;border-radius:12px;border:1px solid #e0e0e0;box-sizing:border-box">
<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px"><div><div style="font-size:1.4em;font-weight:600;color:#333">📝 Multiple Choice Quiz</div><div style="font-size:0.95em;color:#666;margin-top:4px">Knowledge Check</div></div><span style="padding:8px 16px;border-radius:20px;font-weight:600;font-size:0.9em;background:#e3f2fd;color:#1976d2">4 questions</span></div>
<div style="background:#fff;border-radius:12px;border:2px solid #e0e0e0;padding:18px 24px;margin:20px 0 12px 0"><div style="font-size:0.85em;font-weight:600;color:#1976d2;margin-bottom:6px">Question 1 of 4</div><div style="font-size:1.15em;color:#333;line-height:1.6">Which set operation should you use when the two DataFrames have the same columns but in a different order?</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">A</div><div style="flex:1;color:#333">union()</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">B</div><div style="flex:1;color:#333">unionByName()</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">C</div><div style="flex:1;color:#333">intersect()</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">D</div><div style="flex:1;color:#333">subtract()</div></div>
<details style="margin:4px 0 8px 0"><summary style="cursor:pointer;color:#1976d2;font-weight:600;margin:4px 0">Show answer</summary><div style="margin-top:8px;padding:12px 16px;background:#e8f5e9;border:2px solid #4caf50;border-radius:8px"><div style="font-weight:700;color:#2e7d32">✓ B. unionByName()</div><div style="color:#333;margin-top:6px;line-height:1.5">union() matches columns by position, so differing column orders silently combine the wrong columns. unionByName() matches columns by name regardless of order.</div></div></details>
<div style="background:#fff;border-radius:12px;border:2px solid #e0e0e0;padding:18px 24px;margin:20px 0 12px 0"><div style="font-size:0.85em;font-weight:600;color:#1976d2;margin-bottom:6px">Question 2 of 4</div><div style="font-size:1.15em;color:#333;line-height:1.6">When performing a join operation in Spark, which statement is correct?</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">A</div><div style="flex:1;color:#333">Cross joins are the default join type</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">B</div><div style="flex:1;color:#333">Join conditions can only use a single column</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">C</div><div style="flex:1;color:#333">Broadcast hints can optimize joins with small tables</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">D</div><div style="flex:1;color:#333">Joins always require identical column names on both sides</div></div>
<details style="margin:4px 0 8px 0"><summary style="cursor:pointer;color:#1976d2;font-weight:600;margin:4px 0">Show answer</summary><div style="margin-top:8px;padding:12px 16px;background:#e8f5e9;border:2px solid #4caf50;border-radius:8px"><div style="font-weight:700;color:#2e7d32">✓ C. Broadcast hints can optimize joins with small tables</div><div style="color:#333;margin-top:6px;line-height:1.5">broadcast() copies the small DataFrame to every Executor so the large side never shuffles. Inner is the default join type, conditions can use multiple columns, and differing column names are handled with join expressions.</div></div></details>
<div style="background:#fff;border-radius:12px;border:2px solid #e0e0e0;padding:18px 24px;margin:20px 0 12px 0"><div style="font-size:0.85em;font-weight:600;color:#1976d2;margin-bottom:6px">Question 3 of 4</div><div style="font-size:1.15em;color:#333;line-height:1.6">Which is NOT a recommended practice for optimizing join performance in Spark?</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">A</div><div style="flex:1;color:#333">Using broadcast hints for small tables</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">B</div><div style="flex:1;color:#333">Caching frequently joined DataFrames</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">C</div><div style="flex:1;color:#333">Always using cross joins for better distribution</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">D</div><div style="flex:1;color:#333">Repartitioning to handle data skew</div></div>
<details style="margin:4px 0 8px 0"><summary style="cursor:pointer;color:#1976d2;font-weight:600;margin:4px 0">Show answer</summary><div style="margin-top:8px;padding:12px 16px;background:#e8f5e9;border:2px solid #4caf50;border-radius:8px"><div style="font-weight:700;color:#2e7d32">✓ C. Always using cross joins for better distribution</div><div style="color:#333;margin-top:6px;line-height:1.5">A cross join produces every combination of rows - its cost grows multiplicatively and it is almost never a performance optimization. The other three are standard join tuning practices.</div></div></details>
<div style="background:#fff;border-radius:12px;border:2px solid #e0e0e0;padding:18px 24px;margin:20px 0 12px 0"><div style="font-size:0.85em;font-weight:600;color:#1976d2;margin-bottom:6px">Question 4 of 4</div><div style="font-size:1.15em;color:#333;line-height:1.6">Which join type returns only the rows from the left DataFrame that have NO match in the right DataFrame?</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">A</div><div style="flex:1;color:#333">left</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">B</div><div style="flex:1;color:#333">left_semi</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">C</div><div style="flex:1;color:#333">left_anti</div></div>
<div style="display:flex;align-items:center;gap:16px;padding:12px 20px;background:#fff;border:2px solid #e0e0e0;border-radius:8px;margin:0 0 8px 0"><div style="width:32px;height:32px;min-width:32px;border:2px solid #e0e0e0;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;color:#333">D</div><div style="flex:1;color:#333">full_outer</div></div>
<details style="margin:4px 0 8px 0"><summary style="cursor:pointer;color:#1976d2;font-weight:600;margin:4px 0">Show answer</summary><div style="margin-top:8px;padding:12px 16px;background:#e8f5e9;border:2px solid #4caf50;border-radius:8px"><div style="font-weight:700;color:#2e7d32">✓ C. left_anti</div><div style="color:#333;margin-top:6px;line-height:1.5">left_anti is the 'NOT EXISTS' of joins: it keeps left rows without a match and returns only the left columns. left_semi is its opposite, keeping left rows that do have a match.</div></div></details>
</div>

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank" style="font-size: 1em; color: #1B5162; text-decoration: underline">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank" style="text-decoration: underline">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank" style="text-decoration: underline">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank" style="text-decoration: underline">Support</a>

<script> document.getElementById("dbx-year").textContent = new Date().getFullYear(); </script>